In [1]:
import pandas as pd
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib
import os 
import sys 
import cvxpy as cp

# import warnings
# warnings.filterwarnings("ignore")

In [428]:
VAR_DIM_CONSTANT = 96 # 24 hour lookahead - how far do we look ahead while optimizing? This is a safe upper bound since the longest session in SLRP-EV history was 21 hours
COST_DC = 500 # how much the utility charges the station operator for demand charge, in units of cents/kW
DELTA_T = 0.25 # time step in hours
POWER_RATE = 6.6 # max power of chargers in units of kW
FLEXIBILITY_CONSTANT = 0.5 # proportion of power consumed in a regular charging session that the user would have requested if they had chose scheduled

# PG&E time of use tariff from Tugba's simulator, in units of cents/kWh
TOU = np.ones((96,)) * 17.5 # off-peak 0.175  cents / kWh
TOU[64:84] = 36.7 ## 4 pm - 9 pm peak 0.367 cents / kWh
TOU[36:56] = 14.9 ## 9 am - 2 pm super off-peak 0.49 $ / kWh  to cents / kWh
TOU = np.concatenate([TOU, TOU, TOU]) # Repeat TOU in case any charging sessions wrap around to the next day

# Grid search for prices
PRICES = np.arange(20, 1000, 100) # Define the values for z_reg and z_sch
TARIFF_GRID = [(z_sch, z_reg) for z_reg in PRICES for z_sch in PRICES if z_reg >= z_sch] # Create combinations where z_reg >= z_sch

# Discrete choice model
DCM_CHARGING_SCH_PARAMS = np.array([[ - POWER_RATE * 0.0184 / 2], [POWER_RATE * 0.0184 / 2], [0], [0]])
DCM_CHARGING_REG_PARAMS = np.array([[POWER_RATE * 0.0184 / 2], [- POWER_RATE * 0.0184 / 2], [0], [0.341]])
DCM_LEAVING_PARAMS = np.array([[POWER_RATE * 0.005 / 2], [POWER_RATE * 0.005 / 2], [0], [-1]])
THETA = np.vstack((DCM_CHARGING_SCH_PARAMS.T, DCM_CHARGING_REG_PARAMS.T, DCM_LEAVING_PARAMS.T))

MONTE_CARLO = False # If True, re-evaluate user choices with DCM. If False, assume users choose what they did in reality
VERBOSE = True # If True, print information about how the optimization is going

In [429]:
zk = [20, 20, 1, 1]
vk = softmax(THETA @ zk)
vk

array([0.3207057, 0.4510255, 0.2282688])

In [430]:
def get_timestep_info(row, current_time):
    """
    Helper function to convert information from a row in sessions_df to array indices

        row: row from sessions_df
        current_time: time of optimization as a pd.datetime object
    """
    TOU_current_idx = int(np.ceil((current_time.hour + current_time.minute / 60) / DELTA_T)) # current time, beginning of optimization horizon
    start_time = pd.to_datetime(row['startChargeTime'])
    TOU_start_idx = int(np.ceil((start_time.hour + start_time.minute / 60) / DELTA_T))
    TOU_end_idx = int(np.floor(TOU_start_idx + row['DurationHrs'] / DELTA_T)) # end time index
    N_remain = TOU_end_idx - TOU_current_idx # number of timesteps remaining
    return TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain


def get_new_sch_obj(row, z, u):
    """
    Helper function to generate the scheduled objective for the newest EV arrival if they choose scheduled.

        row: row from sessions_df
        z: tuple of (p_sch, p_reg)
        u: cp.Variable for power profile
    """
    TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(row, pd.to_datetime(row['startChargeTime']))
    power_profile = u[: N_remain]
    power_profile = cp.reshape(power_profile, (power_profile.shape[0],)).T
    return power_profile @ (TOU[TOU_start_idx : TOU_end_idx] - z[0]).reshape(-1)

def get_new_reg_obj(row, z):
    """
    Helper function to generate the objective for the newest EV arrival if they choose regular.

        row: row from sessions_df
        z: tuple of (p_sch, p_reg)
    """
    # This code assumes that we know exactly how long the user will charge for regular (don't necessarily know in reality)
    TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(row, pd.to_datetime(row['startChargeTime']))

    if row['choice'] == 'SCHEDULED' and not pd.isna(row['energyReq_Wh']):
        e_need = row['energyReq_Wh'] / 1000 / DELTA_T
    elif row['choice'] == 'SCHEDULED':
        e_need = FLEXIBILITY_CONSTANT * row['cumEnergy_Wh'] / 1000 / DELTA_T
    else:
        e_need = row['cumEnergy_Wh'] / 1000 / DELTA_T

    N_reg = int(e_need // POWER_RATE) # how many time steps would it take the user to charge if they chose regular?
    return np.sum(POWER_RATE * (TOU[TOU_start_idx : TOU_start_idx + N_reg] - z[1]))



def get_power_profile_idx(row, current_time):
    """
    Helper function to get the current index of the power profile (i.e. how many timeseteps has the EV been charging so far)

        row: row from sessions_df
        current_time: time of optimization as a pd.datetime object
    """
    start_time = pd.to_datetime(row['startChargeTime'])
    current_time = (current_time + pd.Timedelta(minutes=15)).floor('15min')
    power_profile_current_idx = int(np.ceil((current_time - start_time).total_seconds() / 3600 / DELTA_T))
    return power_profile_current_idx


def get_e_need(row, current_time, power_profiles):
    """
    Helper function to calculate the energy demand of a particular session.

        row: row from sessions_df
        current_time: time of optimization as a pd.datetime object
        power_profiles: dictionary mapping dcosIds to power_profiles
    """
    if row['choice'] == 'SCHEDULED' and not pd.isna(row['energyReq_Wh']):
        e_need = row['energyReq_Wh'] / 1000 / DELTA_T
    elif row['choice'] == 'SCHEDULED':
        e_need = FLEXIBILITY_CONSTANT * row['cumEnergy_Wh'] / 1000 / DELTA_T
    else:
        e_need = row['cumEnergy_Wh'] / 1000 / DELTA_T
        
    power_profile_current_idx = get_power_profile_idx(row, current_time)
    TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(row, current_time)
    
    if len(power_profiles[row['dcosId']]) > 0:
        e_need -= sum(power_profiles[row['dcosId']][:power_profile_current_idx]) # if user has already consumed some power, subtract it from their demand
        if e_need < 0: # handle numerical imprecision and set completed charging sessions to exactly zero
            e_need = 0
    if e_need  > N_remain * POWER_RATE: # if user requests an infeasible amount of power
        # print(f"{row['dcosId']} infeasible, needed {e_need} in {N_remain} timesteps, a rate of {e_need / N_remain} kW")
        e_need = N_remain * POWER_RATE
    # print(f"{row['dcosId']} needs {e_need} in {N_remain} timesteps, a rate of {e_need / N_remain} kW")
    return e_need

In [431]:
def get_J(u, z, v, p_dc_sch, p_dc_reg, sub_df, current_time, running_peak, power_profiles, prices):
    """
    Helper function to set up the objective function

    Inputs:
        u: cvxpy variable for power profile
        z: array where [tariff_flex, tariff_asap, tariff_overstay, leave = 1 ] (units: cents/kWh)
        v: array with softmax results [sm_c, sm_uc, sm_y] (sm_y = leave) 
        p_dc_sch: cvxpy variable which represents the peak power if the most recent user chooses scheduled
        p_dc_reg: cvxpy variable which represents the peak power if the most recent user chooses regular
        sub_df: dataframe containing rows of sessions_df that represent active sessions at the time of optimization
        current_time: time of optimization
        running_peak: ruunning peak power this billing cycle
        power_profiles: dictionary mapping dcosIds to power_profiles
        prices: dictionary mapping dcodIds to (sch_price, reg_price) tuples
    """
    num_sch_user = 0
    num_reg_user = 0

    existing_sch_obj = cp.Constant(0) # profit objective for existing scheduled
    existing_reg_obj = 0 # profit objective for existing regular
    
    for index, row in sub_df.iloc[:-1].iterrows():
        TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(row, current_time)

        if row['choice'] == 'SCHEDULED':
            price = prices[row['dcosId']][0]
            adj_constant = int((num_sch_user + 1) * VAR_DIM_CONSTANT)
            num_sch_user += 1
            power_profile = u[adj_constant: (adj_constant + N_remain)]
            power_profile = cp.reshape(power_profile, (power_profile.shape[0],)).T
            existing_sch_obj += power_profile @ (TOU[TOU_current_idx : TOU_end_idx] - price).reshape(-1)
            # print('shapes')
            # print(power_profile.shape)
            # print((TOU[TOU_current_idx:TOU_end_idx] - price).reshape(-1).shape)
            # print((u[adj_constant: (adj_constant + N_remain)].T @ (TOU[TOU_current_idx : TOU_end_idx] - price).reshape(-1)).shape)
        else: # Assumes we know exactly how long they will stay
            price = prices[row['dcosId']][1]
            if len(power_profiles[row['dcosId']]) > 0 and TOU_end_idx - TOU_current_idx > 0:
                existing_reg_obj += power_profiles[row['dcosId']][-(TOU_end_idx - TOU_current_idx):] @ (TOU[TOU_current_idx : TOU_end_idx] - price)
            else:
                existing_reg_obj += np.array([POWER_RATE] * N_remain) @ (TOU[TOU_current_idx : TOU_end_idx] - price)
            num_reg_user += 1

    sch_power_sum_profile = cp.reshape(u, (VAR_DIM_CONSTANT, num_sch_user + 1)).T
    sch_power_sum_profile = cp.sum(sch_power_sum_profile, axis=0) # Shape: (self.var_dim_constant,)
    
    last_row = sub_df.iloc[-1]
    new_sch_obj = get_new_sch_obj(last_row, z, u)
    new_reg_obj = get_new_reg_obj(last_row, z)
    new_leave_obj = 0

    current_peak_sch = POWER_RATE * num_reg_user + cp.max(sch_power_sum_profile)
    current_peak_reg =  POWER_RATE * (num_reg_user + 1) + cp.max(cp.sum(cp.reshape(u[VAR_DIM_CONSTANT:], (VAR_DIM_CONSTANT, num_sch_user)).T, axis=0))

    # J0 = ((new_sch_obj + existing_sch_obj + existing_reg_obj) + COST_DC * (p_dc_sch - running_peak)) * v[0]
    # J1 = ((new_reg_obj + existing_sch_obj + existing_reg_obj + COST_DC * (p_dc_reg - running_peak))) * v[1]
    # J2 = (new_leave_obj + existing_sch_obj + existing_reg_obj) * v[2]
    # if not isinstance(existing_sch_obj, int):
        # existing_sch_obj = existing_sch_obj[0, 0]
        # print(existing_sch_obj)
    J0 = (new_sch_obj + existing_sch_obj + existing_reg_obj) * v[0]
    J1 = (new_reg_obj + existing_sch_obj + existing_reg_obj) * v[1]
    J2 = (new_leave_obj + existing_sch_obj + existing_reg_obj) * v[2]
    
    J = J0 + J1 + J2

    return J, [J0 / v[0], J1 / v[1], J2 / v[2], new_sch_obj, new_reg_obj, existing_sch_obj, existing_reg_obj, COST_DC * (p_dc_sch - running_peak)], current_peak_sch, current_peak_reg

In [432]:
def argmin_u(z, v, sub_df, current_time, running_peak, power_profiles, prices):
    """
    Function to minimize charging cost. Flexible charging with variable power schedule
    
        Inputs: 
        z: array where [tariff_flex, tariff_asap, tariff_overstay, leave = 1 ]
        v: array with softmax results [sm_c, sm_uc, sm_y] (sm_y = leave)
        sub_df: dataframe containing rows of sessions_df that represent active sessions at the time of optimization
        current_time: time of optimization
        running_peak: running peak power this billing cycle
        power_profiles: dictionary mapping dcosIds to power_profiles
        prices: dictionary mapping dcodIds to (sch_price, reg_price) tuples
    """
    e_need_lst = []
    N_remain_lst = []
    price_lst = []
    end_charge_times = (pd.to_datetime(sub_df['startChargeTime']) + pd.to_timedelta(sub_df['DurationHrs'], unit='h') - pd.Timedelta(minutes=15)).dt.floor('15min')

    last_row = sub_df.iloc[-1]
    TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(last_row, current_time)
    e_need = get_e_need(last_row, current_time, power_profiles)
    e_need_lst.append(e_need)
    N_remain_lst.append(N_remain)
    
    for index, row in sub_df.iloc[:-1].loc[sub_df['choice'] == 'SCHEDULED'].iterrows():
        TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(row, current_time)
        e_need = get_e_need(row, current_time, power_profiles)
        e_need_lst.append(e_need)
        N_remain_lst.append(N_remain)
      
        if prices[row['dcosId']]:
            price_lst.append(prices[row['dcosId']])
        else:
            price_lst.append(row['sch_centsPerHr'])

    num_sch_user = len(e_need_lst) # number of scheduled users
    
    ### Decision Variables
    e_delivered = cp.Variable(shape = ((VAR_DIM_CONSTANT + 1) * num_sch_user, 1)) # energy delivered
    u = cp.Variable(shape = (VAR_DIM_CONSTANT * num_sch_user, 1)) # charging profile (extra scheduled user profile added in case new user chooses scheduled
    p_dc_sch = cp.Variable(shape = 1)
    p_dc_reg = cp.Variable(shape = 1)    
    
    ### Constraints incorporate all SCH users
    constraints = [u >= 0, u <= POWER_RATE]
    
    # Iterate through all existing flex users
    for i in range(num_sch_user):
        e_need = e_need_lst[i]
        N_remain = N_remain_lst[i]
        
        e_start = int(i * (VAR_DIM_CONSTANT + 1))
        e_end = int(i * (VAR_DIM_CONSTANT + 1) + N_remain)
        e_max = int(i * (VAR_DIM_CONSTANT + 1) + VAR_DIM_CONSTANT)
        u_start = int(i * VAR_DIM_CONSTANT)
        u_end = int(i * VAR_DIM_CONSTANT + N_remain)

        constraints += [cp.sum(u[u_start: u_end]) >= e_need]
        constraints += [u[u_end : u_start + VAR_DIM_CONSTANT] == 0]

    ### Solve
    J, J_array, current_peak_sch, current_peak_reg = get_J(u, z, v, p_dc_sch, p_dc_reg, sub_df, current_time, running_peak, power_profiles, prices)
    
    # Demand charge constraints
    constraints += [running_peak <= p_dc_sch]
    constraints += [running_peak <= p_dc_reg]
    constraints += [current_peak_sch <= p_dc_sch]
    constraints += [current_peak_reg <= p_dc_reg]
    
    obj = cp.Minimize(J)
    prob = cp.Problem(obj, constraints)
    prob.solve()
    if prob.status != 'optimal':
        print(prob.status)
        print("Gurobi failed, cant solve for power")
        prob.solve(solver='GUROBI',verbose=True)
    
    return u.value, e_delivered.value, p_dc_sch.value, p_dc_reg.value, current_peak_sch.value, current_peak_reg.value, J, J_array

In [433]:
from scipy.special import softmax

def grid_search(current_time, running_peak, power_profiles, prices):
    """
    Function to search over a grid of price combinations minimize charging cost.
    
        Inputs: 
        current_time: time of optimization
        running_peak: ruunning peak power this billing cycle
        power_profiles: dictionary mapping dcosIds to power_profiles
        prices: dictionary mapping dcodIds to (sch_price, reg_price) tuples
    """
    sub_df = test_df[pd.to_datetime(test_df['startChargeTime']) <= current_time]
    end_charge_times = (pd.to_datetime(sub_df['startChargeTime']) + pd.to_timedelta(sub_df['DurationHrs'], unit='h') - pd.Timedelta(minutes=15)).dt.floor('15min')
    sub_df = sub_df[end_charge_times >= current_time]
    
    # if after filtering there is no updating to be done, return null result
    if len(sub_df) == 0:
        return None, sub_df
        
    grid_search_results = {}
    for (z_sch_k, z_reg_k) in TARIFF_GRID:
        zk = [z_sch_k, z_reg_k, 1, 1]
        vk = softmax(THETA @ zk).reshape(3,1)
        
        uk_flex, e_delivered, p_dc_sch_k, p_dc_reg_k, current_peak_sch, current_peak_reg, J, J_array  = argmin_u(zk, vk, sub_df, current_time, running_peak, power_profiles, prices)
        grid_search_results[ (z_sch_k, z_reg_k) ] = {}
        grid_search_results[ (z_sch_k, z_reg_k) ]["J"] =  J.value[0]
        grid_search_results[ (z_sch_k, z_reg_k) ]["J_arr"] =  J_array
        grid_search_results[ (z_sch_k, z_reg_k) ]["u"] =  uk_flex
        grid_search_results[ (z_sch_k, z_reg_k) ]["v"] =  vk
        grid_search_results[ (z_sch_k, z_reg_k) ]["p_dc_sch_k"] =  p_dc_sch_k
        grid_search_results[ (z_sch_k, z_reg_k) ]["p_dc_reg_k"] =  p_dc_reg_k
        grid_search_results[ (z_sch_k, z_reg_k) ]["current_peak_sch"] =  current_peak_sch
        grid_search_results[ (z_sch_k, z_reg_k) ]["current_peak_reg"] =  current_peak_reg

    return grid_search_results, sub_df

In [434]:
def simulate(test_df):
    """
    Given a test DataFrame, replay it and simulate the real-time optimization and control decisions

        Inputs:
        test_df: the pandas DataFrame to use in simulation
    """
    power_profiles = {c : [] for c in test_df['dcosId']}
    prices = {c : None for c in test_df['dcosId']}
    running_peak = 0
    
    for startChargeTime in pd.to_datetime(test_df['startChargeTime']):
        grid_search_results, sub_df = grid_search(startChargeTime, running_peak, power_profiles, prices)
    
        min_key = min(grid_search_results, key=lambda k: grid_search_results[k]['J'])
        min_J = grid_search_results[min_key]['J']
        min_J_arr = grid_search_results[min_key]['J_arr']
        u = grid_search_results[min_key]['u']
        v = grid_search_results[min_key]['v']
        dc_sch = grid_search_results[min_key]['p_dc_sch_k']
        dc_reg = grid_search_results[min_key]['p_dc_reg_k']
        current_peak_sch = grid_search_results[min_key]['current_peak_sch']
        current_peak_reg = grid_search_results[min_key]['current_peak_reg']
    
        num_sch_user = 0
        for index, row in sub_df.iloc[:-1].loc[sub_df['choice'] == 'SCHEDULED'].iterrows():
            TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(row, startChargeTime)
            
            adj_constant = int((num_sch_user + 1) * VAR_DIM_CONSTANT)
            num_sch_user += 1
            power_profiles[row['dcosId']][(TOU_current_idx - TOU_start_idx):(TOU_current_idx - TOU_start_idx) + N_remain] = u[adj_constant: (adj_constant + N_remain)].flatten()
    
        last_row = sub_df.iloc[-1]
        prices[last_row['dcosId']] = min_key

        zk = [min_key[0], min_key[1], 1, 1]
        vk = softmax(THETA @ zk).flatten()#.reshape(3,1)
        if MONTE_CARLO:
            normalized_probs = vk[:2] / vk[:2].sum() # only simulate sch/reg choices, no leaving in the simulation
            choice = np.random.choice(['SCHEDULED', 'REGULAR'], p=normalized_probs)
            test_df.loc[test_df['dcosId'] == last_row['dcosId'], 'choice'] = choice
        else:
            choice = last_row['choice']
    
        if choice == 'SCHEDULED':
            running_peak = max(running_peak, dc_sch)
            power_profiles[last_row['dcosId']] = u[:VAR_DIM_CONSTANT].flatten()
            num_sch_user += 1
        else:
            running_peak = max(running_peak, dc_reg)
            TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(last_row, startChargeTime)
            
            N_reg = last_row['cumEnergy_Wh'] / 1000 / POWER_RATE / DELTA_T # how many time steps would it take the user to charge if they chose regular?
            N_reg_remainder = N_reg % 1 # for that last timestep, what fraction of a timestep is charging needed to satisfy demand?
            N_reg = int(N_reg // 1)            
            power_profiles[last_row['dcosId']] = np.zeros(VAR_DIM_CONSTANT)
            power_profiles[last_row['dcosId']][:N_reg] = np.array([POWER_RATE] * N_reg)
            # if N_reg < N_remain and N_reg_remainder > 0:
            #     power_profiles[last_row['dcosId']][N_reg] = POWER_RATE * N_reg_remainder
    
        if VERBOSE:
            print("---------------------------------------------------------------------")
            print('Done with optimization at', startChargeTime)
            print("Optimal prices:", min_key)
            print('Probabilities', vk)
            print('Optimized delivery of', round(sum(u[:VAR_DIM_CONSTANT])[0] / 4, 2), f'kW to session #{last_row["dcosId"]}')
            print('Number of active sessions:', len(sub_df))
            print('Current peak options', np.round(current_peak_sch, 2), np.round(current_peak_reg, 2))
            print('Running DC options', round(dc_sch[0], 2), round(dc_reg[0], 2))
            print('Peak thus far', round(running_peak[0], 2))
            print('Profit options', min_J_arr[0].value, (min_J_arr[1] if isinstance(min_J_arr[1], np.ndarray) else min_J_arr[1].value), (min_J_arr[2] if isinstance(min_J_arr[2], np.ndarray) else min_J_arr[2].value))
            # print('demand charge obj', min_J_arr[6].value)
            print('new_sch_obj', min_J_arr[3].value)
            print('new_reg_obj', min_J_arr[4])
            print('existing_sch_obj', min_J_arr[5] if isinstance(min_J_arr[5], int) else min_J_arr[5].value)
            print('existing_reg_obj', min_J_arr[6])
            print('Profit', min_J)
            print(grid_search_results[(20, 20)]['J'])

            # min_key = (20, 20)
            # min_J = grid_search_results[min_key]['J']
            # min_J_arr = grid_search_results[min_key]['J_arr']
            # u = grid_search_results[min_key]['u']
            # v = grid_search_results[min_key]['v']
            # dc_sch = grid_search_results[min_key]['p_dc_sch_k']
            # dc_reg = grid_search_results[min_key]['p_dc_reg_k']
            # current_peak_sch = grid_search_results[min_key]['current_peak_sch']
            # current_peak_reg = grid_search_results[min_key]['current_peak_reg']
            # zk = [min_key[0], min_key[1], 1, 1]
            # vk = softmax(THETA @ zk).flatten()#.reshape(3,1)
            # print("**********************")
            # print("Optimal prices:", min_key)
            # print('Probabilities', vk)
            # print('Optimized delivery of', round(sum(u[:VAR_DIM_CONSTANT])[0] / 4, 2), f'kW to session #{last_row["dcosId"]}')
            # print('Number of active sessions:', len(sub_df))
            # print('Current peak options', np.round(current_peak_sch, 2), np.round(current_peak_reg, 2))
            # print('Running DC options', round(dc_sch[0], 2), round(dc_reg[0], 2))
            # print('Peak thus far', round(running_peak[0], 2))
            # print('Profit options', min_J_arr[0].value, (min_J_arr[1] if isinstance(min_J_arr[1], np.ndarray) else min_J_arr[1].value), (min_J_arr[2] if isinstance(min_J_arr[2], np.ndarray) else min_J_arr[2].value))
            # # print('demand charge obj', min_J_arr[6].value)
            # print('new_sch_obj', min_J_arr[3].value)
            # print('new_reg_obj', min_J_arr[4])
            # print('existing_sch_obj', min_J_arr[5] if isinstance(min_J_arr[5], int) else min_J_arr[5].value)
            # print('existing_reg_obj', min_J_arr[6])
            # print('Profit', min_J)
            # print(grid_search_results[(20, 20)]['J'])
            # print('v', v)


# [J0, J1, J2, new_sch_obj, new_reg_obj, existing_sch_obj, existing_reg_obj, COST_DC * (p_dc_sch - running_peak)], current_peak_sch, current_peak_reg    

    return power_profiles, prices

In [435]:
def aggregate_power_profiles(test_df, power_profiles):
    """
    Aggregate the power profiles from a simulation

        Inputs:
        test_df: the pandas DataFrame used in the simulation
        power_profiles: dictionary mapping dcosIds to power profiles
    """
    
    filtered_power_profiles = {k: v for k, v in power_profiles.items() if len(v) > 0}
    agg_power_profile = np.zeros(32 * VAR_DIM_CONSTANT)
    
    for key, power_profile in filtered_power_profiles.items():
        matching_row = test_df.loc[test_df['dcosId'] == key]
        row = matching_row.squeeze() 
        start_time = pd.to_datetime(row['startChargeTime'])
        start_of_month = start_time.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
        i = int(np.ceil((start_time - start_of_month).total_seconds() / (15 * 60)))
        agg_power_profile[i : i+len(power_profile)] += power_profile

    return agg_power_profile

In [436]:
def get_profit(test_df, power_profiles, prices):
    """
    Aggregate the profit from a simulation

        Inputs:
        test_df: the pandas DataFrame used in the simulation
        power_profiles: dictionary mapping dcosIds to power_profiles
        prices: dictionary mapping dcodIds to (sch_price, reg_price) tuples
    """
    
    profit = 0
    for index, row in test_df.iterrows():
        current_time = pd.to_datetime(row['startChargeTime'])
        TOU_start_idx, TOU_current_idx, TOU_end_idx, N_remain = get_timestep_info(row, current_time)
        power_profile = power_profiles[row['dcosId']]
        if row['choice'] == 'SCHEDULED':
            profit += np.sum(power_profile[:N_remain] * (prices[row['dcosId']][0] - TOU[TOU_start_idx : TOU_end_idx]))
        else:
            profit += np.sum(power_profile[:N_remain] * (prices[row['dcosId']][1] - TOU[TOU_start_idx : TOU_end_idx]))
            
    return profit

In [437]:
sessions_df = pd.read_csv("/Users/sam/Desktop/StationLevelPowerForecasting/data/Sessions3.csv")
sessions_df = sessions_df.sort_values(by='startChargeTime')

In [438]:
test_df = sessions_df[(pd.to_datetime(sessions_df['connectTime']).dt.year == 2024) & (pd.to_datetime(sessions_df['connectTime']).dt.month == 1)]
test_df['choice'] = 'SCHEDULED'
test_df = test_df[test_df['DurationHrs'] > 0.5]
test_df = test_df[test_df['cumEnergy_Wh'] > 0]
power_profiles, prices = simulate(test_df)
agg_power_profile_all_sch = aggregate_power_profiles(test_df, power_profiles)
profit_all_sch = get_profit(test_df, power_profiles, prices)
profit_all_sch - COST_DC * max(agg_power_profile_all_sch), max(agg_power_profile_all_sch)

/var/folders/_j/vd_r650147l_5pl4bwls0xg00000gn/T/ipykernel_97509/1478677460.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['choice'] = 'SCHEDULED'


---------------------------------------------------------------------
Done with optimization at 2024-01-02 10:11:14
Optimal prices: (520, 920)
Probabilities [8.21713502e-01 9.25818998e-22 1.78286498e-01]
Optimized delivery of 34.65 kW to session #6671
Number of active sessions: 1
Current peak options 6.6 6.6
Running DC options 15.51 10.69
Peak thus far 15.51
Profit options [-69903.90000746] [-59736.6] [0.]
new_sch_obj -69903.9000074607
new_reg_obj -59736.600000000006
existing_sch_obj 0.0
existing_reg_obj 0
Profit -57440.978469451664
-345.4893556617274
---------------------------------------------------------------------
Done with optimization at 2024-01-03 00:58:06
Optimal prices: (520, 920)
Probabilities [8.21713502e-01 9.25818998e-22 1.78286498e-01]
Optimized delivery of 19.8 kW to session #6674
Number of active sessions: 1
Current peak options 6.6 6.6
Running DC options 18.9 19.6
Peak thus far 18.9
Profit options [-39798.00000596] [-29782.5] [0.]
new_sch_obj -39798.000005961105
new_

/opt/anaconda3/envs/slrpev/lib/python3.10/site-packages/cvxpy/problems/problem.py:1296: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


---------------------------------------------------------------------
Done with optimization at 2024-01-04 10:04:15
Optimal prices: (520, 920)
Probabilities [8.21713502e-01 9.25818998e-22 1.78286498e-01]
Optimized delivery of 3.3 kW to session #6679
Number of active sessions: 3
Current peak options 19.8 19.8
Running DC options 56.46 69.37
Peak thus far 56.46
Profit options [-152060.04000092] [-151366.38000086] [-145392.72000086]
new_sch_obj -6667.320000061106
new_reg_obj -5973.66
existing_sch_obj -145392.72000085952
existing_reg_obj 0
Profit -150871.34686619294
-145429.4914265442


KeyboardInterrupt: 

In [295]:
profit_all_sch, COST_DC * max(agg_power_profile_all_sch)

NameError: name 'profit_all_sch' is not defined

In [18]:
test_df = sessions_df[(pd.to_datetime(sessions_df['connectTime']).dt.year == 2024) & (pd.to_datetime(sessions_df['connectTime']).dt.month == 1)]
test_df['choice'] = 'REGULAR'
test_df = test_df[test_df['DurationHrs'] > 0.5]
test_df = test_df[test_df['cumEnergy_Wh'] > 0]
power_profiles, prices = simulate(test_df)
agg_power_profile_all_reg = aggregate_power_profiles(test_df, power_profiles)
profit_all_reg = get_profit(test_df, power_profiles, prices)
profit_all_reg - COST_DC * max(agg_power_profile_all_reg), max(agg_power_profile_all_reg)

(335913.5300000001, 46.2)

In [19]:
test_df = sessions_df[(pd.to_datetime(sessions_df['connectTime']).dt.year == 2024) & (pd.to_datetime(sessions_df['connectTime']).dt.month == 1)]
test_df = test_df[test_df['DurationHrs'] > 0.5]
test_df = test_df[test_df['cumEnergy_Wh'] > 0]
power_profiles, prices = simulate(test_df)
agg_power_profile = aggregate_power_profiles(test_df, power_profiles)
profit = get_profit(test_df, power_profiles, prices)
profit - COST_DC * max(agg_power_profile), max(agg_power_profile_all_reg)

(345209.2913517962, 46.2)

In [16]:
import plotly.graph_objects as go
import plotly.io as pio

# Create the figure
fig = go.Figure()

# Add each time series with a distinct color
fig.add_trace(go.Scatter(y=agg_power_profile_all_sch, mode='lines', name='All Scheduled Profile', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=agg_power_profile_all_reg, mode='lines', name='All Regular Profile', line=dict(color='green')))
fig.add_trace(go.Scatter(y=agg_power_profile, mode='lines', name='Original Profile', line=dict(color='red')))

# Customize the layout
fig.update_layout(
    title="Power Profiles",
    xaxis_title="Index",
    yaxis_title="Power",
    legend_title="Profile",
)

# Display in the browser
pio.show(fig, renderer="browser")